# Notebook 04: Naïve Bayes, Logistic Regression & Support Vector Machines

**Capstone Stage 1 | Modules 11, 13, 14**  
**Dataset:** Polish Companies Bankruptcy (UCI ID 365)  
**Author:** Srini | Imperial College London — Professional Certificate in ML & AI

---

## Objectives

- Apply **Naïve Bayes** with Laplace smoothing to the credit dataset; evaluate feasibility given feature dependencies (Module 11)
- Build and tune **Logistic Regression** — decision threshold optimisation, coefficient interpretation (Module 13)
- Implement **Support Vector Machines** with RBF and linear kernels; soft-margin tuning (Module 14)
- Compare all three against ensemble baselines from Notebook 03
- Discuss why each method's assumptions are satisfied (or violated) for credit risk classification

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_style('whitegrid')
np.random.seed(42)

# Load data
df = pd.read_csv('../data/polish_bankruptcy.csv')
with open('../data/feature_map.json') as f:
    feature_map = json.load(f)
feature_cols = [c for c in df.columns if c.startswith('X')]
X_raw = df[feature_cols].values
y = df['target'].values

X_trainval, X_test, y_trainval, y_test = train_test_split(X_raw, y, test_size=0.20, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=0.25, random_state=42, stratify=y_trainval)

preprocessor = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', RobustScaler())])
X_train_pp = preprocessor.fit_transform(X_train)
X_val_pp   = preprocessor.transform(X_val)
X_test_pp  = preprocessor.transform(X_test)

smote = SMOTE(random_state=42, k_neighbors=5)
X_train_sm, y_train_sm = smote.fit_resample(X_train_pp, y_train)
print('Data ready. Shapes — Train (SMOTE):', X_train_sm.shape, '| Val:', X_val_pp.shape)

---
## 1. Naïve Bayes — Module 11

**Key assumption:** Features are conditionally independent given the class label. For financial ratios, this is a strong and almost certainly violated assumption — many ratios are definitionally correlated (e.g., leverage ratio and interest coverage). We test Naïve Bayes anyway and quantify the performance penalty from violated assumptions.

In [ ]:
# Gaussian Naïve Bayes — assumes features are normally distributed per class
# For financial ratios this is an approximation; log-transform helps but we test raw first

gnb = GaussianNB()
gnb.fit(X_train_sm, y_train_sm)
gnb_proba = gnb.predict_proba(X_val_pp)[:, 1]
gnb_auc = roc_auc_score(y_val, gnb_proba)
gnb_f1  = f1_score(y_val, (gnb_proba >= 0.5).astype(int))

print('Gaussian Naïve Bayes Results:')
print(f'  Val AUC-ROC : {gnb_auc:.4f}')
print(f'  Val F1 (bankrupt): {gnb_f1:.4f}')
print(classification_report(y_val, (gnb_proba>=0.5).astype(int), target_names=['Solvent','Bankrupt']))

# Visualise class-conditional distributions for key features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
key_feats = [0, 1, 2]  # X1, X2, X3 (indices)
feat_labels = ['Net Profit / Total Assets (X1)', 'Total Liabilities / Total Assets (X2)', 'Working Capital / Total Assets (X3)']
COLOURS = {'bankrupt': '#d62728', 'solvent': '#1f77b4'}

for ax, feat_idx, label in zip(axes, key_feats, feat_labels):
    # Gaussian NB learns mean and var per class — plot fitted Gaussian
    x_range = np.linspace(-3, 3, 200)
    for cls, cname, colour in [(0, 'Solvent', COLOURS['solvent']), (1, 'Bankrupt', COLOURS['bankrupt'])]:
        mean = gnb.theta_[cls, feat_idx]
        var  = gnb.var_[cls, feat_idx]
        pdf  = (1/np.sqrt(2*np.pi*var)) * np.exp(-0.5*(x_range-mean)**2/var)
        ax.plot(x_range, pdf, color=colour, linewidth=2, label=f'{cname} (μ={mean:.2f})')
        # Actual histogram
        ax.hist(X_train_pp[y_train==cls, feat_idx], bins=40, density=True,
                alpha=0.25, color=colour)
    ax.set_title(label.split('(')[0].strip(), fontsize=9, fontweight='bold')
    ax.set_xlabel('Standardised value'); ax.legend(fontsize=8)
    ax.set_xlim(-3, 3)

plt.suptitle('Naïve Bayes: Fitted Gaussians vs Actual Distributions\n(Departure from Gaussianity is a model limitation)',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/04_naive_bayes_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nAudit note: Naïve Bayes assumes feature independence and Gaussian distributions.')
print('Both assumptions are violated for correlated financial ratios.')
print('Performance reflects these violated assumptions — suitable only as a fast baseline.')

---
## 2. Logistic Regression — Threshold Tuning & Coefficient Interpretation (Module 13)

In [ ]:
# Logistic Regression with L2 regularisation
lr = LogisticRegression(C=0.1, max_iter=500, random_state=42, class_weight='balanced')
lr.fit(X_train_sm, y_train_sm)
lr_proba = lr.predict_proba(X_val_pp)[:,1]
lr_auc   = roc_auc_score(y_val, lr_proba)
print(f'Logistic Regression (C=0.1) Val AUC: {lr_auc:.4f}')

# Decision threshold optimisation — find threshold maximising F1 on validation set
thresholds = np.linspace(0.01, 0.99, 100)
f1_scores = [f1_score(y_val, (lr_proba >= t).astype(int), zero_division=0) for t in thresholds]
best_thresh = thresholds[np.argmax(f1_scores)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 vs threshold
axes[0].plot(thresholds, f1_scores, color='#2196F3', linewidth=2)
axes[0].axvline(best_thresh, color='red', linestyle='--', label=f'Best threshold={best_thresh:.2f}')
axes[0].set_xlabel('Decision Threshold'); axes[0].set_ylabel('F1 Score (Bankrupt class)')
axes[0].set_title('Decision Threshold Optimisation\n(Maximise F1 on Validation Set)', fontweight='bold')
axes[0].legend()

# Top LR coefficients — credit analyst interpretation
coefs = lr.coef_[0]
top_pos_idx = np.argsort(coefs)[::-1][:8]
top_neg_idx = np.argsort(coefs)[:8]
top_idx_combined = np.concatenate([top_neg_idx, top_pos_idx])
labels = [feature_map.get(f'X{i+1}', f'X{i+1}').replace('_',' ')[:35] for i in top_idx_combined]
colours = ['#d62728' if c > 0 else '#1f77b4' for c in coefs[top_idx_combined]]

axes[1].barh(range(len(top_idx_combined)), coefs[top_idx_combined], color=colours, alpha=0.8)
axes[1].set_yticks(range(len(top_idx_combined)))
axes[1].set_yticklabels(labels, fontsize=7)
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_xlabel('Log-odds coefficient')
axes[1].set_title('Logistic Regression Coefficients\n(Red = bankruptcy risk ↑, Blue = ↓)', fontweight='bold')

plt.suptitle('Logistic Regression — Threshold Tuning & Coefficient Interpretation', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/04_logistic_regression.png', dpi=150, bbox_inches='tight')
plt.show()

lr_pred_opt = (lr_proba >= best_thresh).astype(int)
print(f'\nWith optimised threshold={best_thresh:.2f}:')
print(classification_report(y_val, lr_pred_opt, target_names=['Solvent','Bankrupt']))

---
## 3. Support Vector Machines — Kernel Functions & Soft Margin (Module 14)

In [ ]:
# SVM with RBF kernel — note: SVM is computationally expensive on full 10k dataset
# Use a stratified subsample for training; evaluate on full validation set
from sklearn.utils import resample

# Stratified subsample: 2000 for SVM training (memory/speed)
n_sub = 2000
idx_sub = np.concatenate([
    np.random.choice(np.where(y_train_sm==0)[0], int(n_sub*0.5), replace=False),
    np.random.choice(np.where(y_train_sm==1)[0], int(n_sub*0.5), replace=False)
])
X_svm_train = X_train_sm[idx_sub]
y_svm_train = y_train_sm[idx_sub]

svm_results = []
for kernel, C_values in [('linear', [0.01, 0.1, 1.0]), ('rbf', [0.1, 1.0, 10.0])]:
    for C in C_values:
        svm = SVC(kernel=kernel, C=C, probability=True, random_state=42, class_weight='balanced')
        svm.fit(X_svm_train, y_svm_train)
        proba = svm.predict_proba(X_val_pp)[:,1]
        svm_results.append({
            'kernel': kernel, 'C': C,
            'val_auc': roc_auc_score(y_val, proba),
            'val_f1': f1_score(y_val, (proba>=0.5).astype(int))
        })
        print(f'  SVM {kernel:6s} C={C:5.2f} → AUC={svm_results[-1]["val_auc"]:.4f}  F1={svm_results[-1]["val_f1"]:.4f}')

svm_df = pd.DataFrame(svm_results)
best_svm = svm_df.loc[svm_df['val_auc'].idxmax()]
print(f'\nBest SVM: kernel={best_svm["kernel"]}, C={best_svm["C"]} → AUC={best_svm["val_auc"]:.4f}')

In [ ]:
# Visualise SVM kernel comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for kernel, colour, ax in [('linear', '#9C27B0', axes[0]), ('rbf', '#FF5722', axes[1])]:
    sub = svm_df[svm_df['kernel']==kernel]
    ax.bar([f'C={c}' for c in sub['C']], sub['val_auc'], color=colour, alpha=0.8, edgecolor='black')
    ax.set_title(f'SVM ({kernel.upper()} kernel) — C vs Validation AUC', fontweight='bold')
    ax.set_ylabel('AUC-ROC'); ax.set_ylim(0.5, 1.0)
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.text(i, row['val_auc']+0.005, f'{row["val_auc"]:.3f}', ha='center', fontsize=9)

plt.suptitle('SVM Kernel & Soft-Margin (C) Tuning — Polish Bankruptcy Dataset',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/04_svm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('Key observation: RBF kernel outperforms linear kernel — the bankruptcy decision')
print('boundary is non-linear in feature space, as expected for financial ratios.')
print('Higher C (less regularisation) allows more complex boundaries but risks overfitting.')

---
## 4. Full Module Comparison Summary

In [ ]:
# Comprehensive model comparison across all Stage 1 modules
best_svm_model = SVC(kernel=best_svm['kernel'], C=best_svm['C'],
                     probability=True, random_state=42, class_weight='balanced')
best_svm_model.fit(X_svm_train, y_svm_train)
svm_proba = best_svm_model.predict_proba(X_val_pp)[:,1]

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight='balanced', random_state=42, n_jobs=-1)
rf.fit(X_train_sm, y_train_sm)
rf_proba = rf.predict_proba(X_val_pp)[:,1]

gb = GradientBoostingClassifier(n_estimators=200, max_depth=4, learning_rate=0.05, subsample=0.8, random_state=42)
gb.fit(X_train_sm, y_train_sm)
gb_proba = gb.predict_proba(X_val_pp)[:,1]

all_models = [
    ('Naïve Bayes (Module 11)', gnb_proba),
    ('Logistic Regression (Module 13)', lr_proba),
    (f'SVM {best_svm["kernel"].upper()} C={best_svm["C"]} (Module 14)', svm_proba),
    ('Random Forest (Module 10)', rf_proba),
    ('Gradient Boosting (Module 10)', gb_proba),
]

fig, ax = plt.subplots(figsize=(10, 6))
colours_all = ['#FF9800', '#9C27B0', '#FF5722', '#2196F3', '#F44336']
for (name, proba), colour in zip(all_models, colours_all):
    auc = roc_auc_score(y_val, proba)
    fpr, tpr, _ = roc_curve(y_val, proba)
    ax.plot(fpr, tpr, color=colour, linewidth=2, label=f'{name} (AUC={auc:.3f})')
ax.plot([0,1],[0,1],'k--', alpha=0.3, label='Random')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('Stage 1 Complete Model Comparison — All Modules\nPolish Bankruptcy Early Warning System', fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('../reports/04_all_models_roc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFinal Stage 1 Model Leaderboard:')
print(f'{"Model":<45} {"Val AUC":>10} {"Val F1":>8}')
print('-' * 65)
for name, proba in sorted(all_models, key=lambda x: roc_auc_score(y_val, x[1]), reverse=True):
    auc = roc_auc_score(y_val, proba)
    f1  = f1_score(y_val, (proba>=0.5).astype(int))
    print(f'{name:<45} {auc:>10.4f} {f1:>8.4f}')

---
## 5. Assumptions Audit — Why Do Some Models Underperform?

| Model | Key Assumption | Violated for Credit Data? | Impact |
|-------|---------------|--------------------------|--------|
| Naïve Bayes | Feature independence + Gaussianity | Yes — ratios are correlated; distributions are skewed | AUC penalty |
| Logistic Regression | Linear decision boundary | Partially — non-linear relationships exist | Moderate penalty |
| SVM (Linear) | Linearly separable (in original space) | Yes | Lower AUC |
| SVM (RBF) | Non-linear boundary (kernel trick) | No | Better AUC |
| Random Forest | No parametric assumption | N/A | Strong performance |
| Gradient Boosting | No parametric assumption | N/A | Best performance |

**Conclusion:** Ensemble tree methods dominate because they make no distributional assumptions and naturally handle correlated, skewed financial ratios with missing values. This justifies XGBoost as the primary model in the Audit Toolkit (Notebook 05).

---
## Next Notebook

→ **Notebook 05:** XGBoost Model Training, SHAP Explainability & Audit Diagnostics (Audit Toolkit Core)